In [ ]:
# start_welding_rods.ipynb

import os
import sys

import numpy as np
import serial


def initialize_datafile(number_runs):
    folder_path = os.path.dirname(os.path.realpath("__file__"))
    file_path = os.path.join(folder_path, datafile_name)

    # Delete existing datafile if it exists
    if os.path.isfile(file_path):
        os.remove(file_path)

    # Pre-create rows having a zero count to be a placeholder for each run
    counts = [0] * (number_runs + 1)

    # Save initialized datafile
    np.savetxt(file_path, counts, fmt="%i")
    print(f'Datafile "{datafile_name}" initialized.')


def save_geiger_count(datafile_name, run_number):
    print(f"Measuring counts (decay events) over 30 seconds...")

    # Connect to the MCU based upon local OS
    port = "COM5"
    if sys.platform == "linux":
        port = "/dev/ttyUSB0"
    if sys.platform == "darwin":
        port = "/dev/tty.usbserial-110"
    ser = serial.Serial(port, 115200, 8, "N", 1, timeout=120)

    # Send MCU the command to (r)un the experiment
    ser.write(b"run_geiger_counter\n")

    # Wait until the MCU responds with the detector count
    count = int(ser.readline().strip())

    # Adjust count units to be per minute
    count = count * 2

    # Close the serial port connection to the MCU
    ser.close()

    # Display the count on screen
    print(f"Run # {run_number}, counts per minute = {count:>5,}")

    # Determine the path to the CSV file
    folder_path = os.path.dirname(os.path.realpath("__file__"))
    file_path = os.path.join(folder_path, datafile_name)

    # Update datafile with count for this run
    counts = np.genfromtxt(f"{file_path}", delimiter=",")
    counts[run_number] = count
    np.savetxt(file_path, counts, fmt="%i")


datafile_name = "counts_per_rod.csv"
initialize_datafile(number_runs=6)

In [ ]:
# Run 01 - Insert rods 1 & 2 into BOTTOM row
save_geiger_count(datafile_name, run_number=1)

In [ ]:
# Run 02 - Insert rods 3 & 4 into BOTTOM row
save_geiger_count(datafile_name, run_number=2)

In [ ]:
# Run 03 - Insert rods 5 & 6 into BOTTOM row
save_geiger_count(datafile_name, run_number=3)

In [ ]:
# Run 04 - Insert rods 7 & 8 into TOP row
save_geiger_count(datafile_name, run_number=4)

In [ ]:
# Run 05 - Insert rods 9 & 10 into TOP row
save_geiger_count(datafile_name, run_number=5)

In [ ]:
# Run 06 - Insert rods 11 & 12 into TOP row
save_geiger_count(datafile_name, run_number=6)

In [ ]:
# Determine the relationship bewteen radiation absorption vs. number of rods
%run ./plot_counts_per_rod.ipynb